In [4]:
!python -m pip install optuna

In [6]:
#!/usr/bin/env python3
"""
Optuna‑driven tuning for scikit‑learn MLPClassifier on ai4i2020.csv
==================================================================
• RepeatedStratifiedKFold (5×2) as inner CV  ➜ more stable MCC
• Bayesian optimisation (TPE) with median‑pruning
• Wide search space incl. lbfgs, sgd & adam β₁/β₂
• Early‑stopping enabled where supported
"""

import warnings, sys, os
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import optuna

from sklearn.compose        import ColumnTransformer
from sklearn.model_selection import (
    train_test_split, RepeatedStratifiedKFold, cross_val_score
)
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import matthews_corrcoef, make_scorer
from sklearn.neural_network  import MLPClassifier

RND   = 42
SCORER = make_scorer(matthews_corrcoef)

# ────────────────────────────────────────────────────────────────────
# 1. data load & balancing
# ────────────────────────────────────────────────────────────────────
CSV = "ai4i2020.csv"
if not os.path.exists(CSV):
    sys.exit(f"❌  {CSV} not found")

df = pd.read_csv(CSV)
FEATURES = [
    "Air temperature [K]", "Process temperature [K]",
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"
]
X, y = df[FEATURES], df["Machine failure"]

try:
    from imblearn.under_sampling import RandomUnderSampler
    X_bal, y_bal = RandomUnderSampler(random_state=RND).fit_resample(X, y)
except ImportError:
    min_n = y.value_counts().min()
    df_bal = (df.groupby("Machine failure", group_keys=False)
                .apply(lambda d: d.sample(min_n, random_state=RND))
                .sample(frac=1, random_state=RND))
    X_bal, y_bal = df_bal[FEATURES], df_bal["Machine failure"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_bal, y_bal, test_size=0.2, stratify=y_bal, random_state=RND
)

preproc = ColumnTransformer([("std", StandardScaler(), FEATURES)])

# ────────────────────────────────────────────────────────────────────
# 2. Optuna search space
# ────────────────────────────────────────────────────────────────────
def suggest_architecture(trial):
    depth = trial.suggest_int("n_layers", 1, 3)                      # 1‑3 layers
    return tuple(
        trial.suggest_int(f"n_units_l{i}", 32, 256, step=32)         # 32‑256 units
        for i in range(depth)
    )

def objective(trial):
    solver = trial.suggest_categorical("solver", ["adam", "sgd", "lbfgs"])

    params = dict(
        hidden_layer_sizes = suggest_architecture(trial),
        activation         = trial.suggest_categorical("activation", ["relu", "tanh"]),
        alpha              = trial.suggest_loguniform("alpha", 1e-6, 1e-1),
        solver             = solver,
        random_state       = RND,
        max_iter           = 400,
        batch_size         = trial.suggest_categorical("batch_size", [8, 16, 32, 64, 128]),
    )

    # solver‑specific knobs
    if solver == "adam":
        params.update(
            learning_rate_init = trial.suggest_loguniform("lr_init", 1e-4, 1e-2),
            beta_1             = trial.suggest_float("beta1", 0.8, 0.99),
            beta_2             = trial.suggest_float("beta2", 0.9, 0.999),
            early_stopping     = True,
            n_iter_no_change   = 10,
            validation_fraction= 0.15,
        )
    elif solver == "sgd":
        params.update(
            learning_rate      = trial.suggest_categorical("lr_schedule",
                                                           ["constant", "adaptive", "invscaling"]),
            learning_rate_init = trial.suggest_loguniform("lr_init", 1e-4, 1e-1),
            momentum           = trial.suggest_float("momentum", 0.5, 0.95),
            nesterovs_momentum = True,
            early_stopping     = True,
            n_iter_no_change   = 10,
            validation_fraction= 0.15,
        )

    clf = MLPClassifier(**params)
    pipe = Pipeline([("prep", preproc), ("clf", clf)])

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RND)
    scores = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring=SCORER, n_jobs=-1)
    return scores.mean()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RND),
    pruner = optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=0)
)

print("🔍  Optimising … (≈2–3 min on 8‑core CPU for 100 trials)")
study.optimize(objective, n_trials=100, timeout=None, show_progress_bar=True)

print("\n🏅  Best trial:")
best = study.best_trial
print("CV‑MCC :", round(best.value, 4))
for k, v in best.params.items():
    print(f"  {k:15s}: {v}")

# ────────────────────────────────────────────────────────────────────
# 3. Fit best model on *all* training data & test on hold‑out set
# ────────────────────────────────────────────────────────────────────
final_clf = MLPClassifier(
    hidden_layer_sizes = best.params["hidden_layer_sizes"],
    activation         = best.params["activation"],
    alpha              = best.params["alpha"],
    solver             = best.params["solver"],
    batch_size         = best.params["batch_size"],
    random_state       = RND,
    max_iter           = 400,
    # conditional hyper‑parameters
    **{k:v for k,v in best.params.items() if k not in
       ["hidden_layer_sizes","activation","alpha","solver","batch_size"]}
)

pipe = Pipeline([("prep", preproc), ("clf", final_clf)])
pipe.fit(X_tr, y_tr)

from sklearn.metrics import classification_report
y_pred  = pipe.predict(X_te)
mcc_te  = matthews_corrcoef(y_te, y_pred)
print(f"\n📈  Hold‑out MCC : {mcc_te:.4f}\n")
print(classification_report(y_te, y_pred))

# Optional: unbiased outer CV on all balanced data
from sklearn.model_selection import cross_val_score
outer_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RND)
outer = cross_val_score(pipe, X_bal, y_bal, cv=outer_cv, scoring=SCORER, n_jobs=-1)
print("Outer CV MCCs:", np.round(outer, 4))
print(f"Mean ± SD    : {outer.mean():.4f} ± {outer.std():.4f}")


[I 2025-07-23 18:30:04,910] A new study created in memory with name: no-name-e3093d05-ed6a-44df-b54e-2d2994dcfb7b


🔍  Optimising … (≈2–3 min on 8‑core CPU for 100 trials)


  0%|          | 0/100 [00:00<?, ?it/s]

/var/folders/sq/y62h0qk17yn1y5z20t900rsc0000gn/T/ipykernel_81025/1226062145.py:76: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  alpha              = trial.suggest_loguniform("alpha", 1e-6, 1e-1),
/var/folders/sq/y62h0qk17yn1y5z20t900rsc0000gn/T/ipykernel_81025/1226062145.py:97: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate_init = trial.suggest_loguniform("lr_init", 1e-4, 1e-1),


[I 2025-07-23 18:30:05,929] Trial 0 finished with value: 0.3804624142340645 and parameters: {'solver': 'sgd', 'n_layers': 2, 'n_units_l0': 64, 'n_units_l1': 64, 'activation': 'tanh', 'alpha': 0.0010129197956845735, 'batch_size': 32, 'lr_schedule': 'invscaling', 'lr_init': 0.003752055855124282, 'momentum': 0.6943752583889521}. Best is trial 0 with value: 0.3804624142340645.


...

/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optim

[I 2025-07-23 18:32:16,006] Trial 97 finished with value: 0.8407051117006311 and parameters: {'solver': 'lbfgs', 'n_layers': 2, 'n_units_l0': 224, 'n_units_l1': 96, 'activation': 'relu', 'alpha': 0.0441244263801088, 'batch_size': 128}. Best is trial 95 with value: 0.8539208484717167.


/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optim

[I 2025-07-23 18:32:16,930] Trial 98 finished with value: 0.8462993249406585 and parameters: {'solver': 'lbfgs', 'n_layers': 2, 'n_units_l0': 256, 'n_units_l1': 64, 'activation': 'relu', 'alpha': 0.03217213652453122, 'batch_size': 128}. Best is trial 95 with value: 0.8539208484717167.
[I 2025-07-23 18:32:17,013] Trial 99 finished with value: 0.6845259543309484 and parameters: {'solver': 'sgd', 'n_layers': 2, 'n_units_l0': 96, 'n_units_l1': 64, 'activation': 'relu', 'alpha': 0.02055199908566114, 'batch_size': 128, 'lr_schedule': 'constant', 'lr_init': 0.07325615969529421, 'momentum': 0.8517509026859036}. Best is trial 95 with value: 0.8539208484717167.

🏅  Best trial:
CV‑MCC : 0.8539
  solver         : lbfgs
  n_layers       : 2
  n_units_l0     : 256
  n_units_l1     : 32
  activation     : relu
  alpha          : 0.04376897443216623
  batch_size     : 128


KeyError: 'hidden_layer_sizes'